In [1]:
%pip install -e ./orchestrator
! ansible-galaxy collection install -r orchestrator/ansible/requirements.yaml

import orchestrator
import shutil
assert shutil.which("ansible-playbook")
print(orchestrator.__file__)

Obtaining file:///Users/yassiramami/Downloads/temp_5gbp/orchestrator
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for orchestrator (pyproject.toml) ... done
  Created wheel for orchestrator: filename=orchestrator-0.0.1-py3-none-any.whl size=2360 sha256=8a1ebd58f7ea3c2019e4d7f9aa72210ac651b75b8475384b74051959c9e0c446
  Stored in directory: /private/var/folders/wn/8gl78s7j64vdbp1z9b717xzr0000gn/T/pip-ephem-wheel-cache-92ikav0u/wheels/bd/ba/57/a96b475c1612822303f6f950db393d09764dd5201fc8f34a61
Successfully built orchestrator
  Attempting uninstall: orchestrator
    Found existing installation: orchestrator 0.0.1
    Uninstalling orchestrator-0.0.1:
      Successfully uninstalled orchestrator-0.0.1

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from orchestrator.models import MetaModel
from orchestrator.models.meta.ric import RICComponentType, RICImplementation

metamodel = MetaModel.from_yaml_file("metamodel_osc_ric.yaml")
osc = metamodel.rics["osc"]
assert osc.implementation is RICImplementation.ORAN_SC
assert {component.type for component in osc.components.values()} == set(RICComponentType) - {RICComponentType.RIC}
assert metamodel.ran.components["gnb"].config.ric == "osc"
metamodel.to_yaml_file("metamodel_1.yaml")
metamodel.to_yaml_file("metamodel_full_1.yaml", exclude_defaults=False)

In [3]:
from orchestrator.models import DeploymentModel

deployment_model = DeploymentModel.model_validate(
    {
        "params": {
            "provisioning": {
                "experiment": "ric", # slices experiment
                "extra_path": "/Users/yassiramami/slices/bin",
            },
            "ansible": {
                "roles_path": "/Users/yassiramami/Downloads/temp_5gbp/orchestrator/ansible/roles",
            }
        }
    })
deployment_model.to_yaml_file("deployment_model_1.yaml")
deployment_model.to_yaml_file("deployment_model_full_1.yaml", exclude_defaults=False)

In [4]:
from orchestrator.deployment import BaseBlueprintManagement

reconciled_metamodel, reconciled_deployment_model = BaseBlueprintManagement(metamodel=metamodel, deployment_model=deployment_model).output()

In [5]:
reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_1.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_1.yaml")

In [6]:
from orchestrator.deployment import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [7]:
await prov.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/node/bi/provision : Select project 'post5g-beta'] ***********

TASK [provisioning/node/bi/provision : Select project 'post5g-beta'] ***********

In [8]:
metamodel, deployment_model = prov.output()

deployment_model.to_yaml_file("deployment_model_2.yaml")
metamodel.to_yaml_file("metamodel_2.yaml")

# checkpoint 1: provisioned

In [9]:
from orchestrator.deployment import ProvisioningManagement
from orchestrator.models import DeploymentModel, MetaModel
from orchestrator.models.meta.ric import RICComponent, RICComponentType, RICImplementation

deployment_model = DeploymentModel.from_yaml_file("deployment_model_2.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_2.yaml")

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

In [10]:
prov.get_merged(output="yaml", path="merged_model.yaml")

In [11]:
prov.get_ansible_inventory(path="inventory.ini")

In [12]:
from orchestrator.deployment import K8SSetupManagement

k8s = K8SSetupManagement(metamodel=metamodel, deployment_model=deployment_model)

In [13]:
await k8s.deploy()


PLAY [node3] *******************************************************************
PLAY [node2] *******************************************************************


PLAY [node1] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node2'] ****************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node3'] ****************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node1'] ****************
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node1]

TASK [k8s/prepare : Install base packages required for Kubernetes on 'node1'] ***
[WARNING]: Host 'node3' is using the discovered Python interpreter at '/usr/bin/python3.12

In [14]:
metamodel, deployment_model = k8s.output()

deployment_model.to_yaml_file("deployment_model_3.yaml")
metamodel.to_yaml_file("metamodel_3.yaml")

# checkpoint 2: k8s ready

In [15]:
from orchestrator.deployment import CoreManagement
from orchestrator.models import DeploymentModel, MetaModel

deployment_model = DeploymentModel.from_yaml_file("deployment_model_3.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_3.yaml")

core = CoreManagement(metamodel=metamodel, deployment_model=deployment_model)

In [16]:
reconciled_metamodel, reconciled_deployment_model = core.output()

reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_2.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_2.yaml")

In [17]:
core.mutatable_metamodel.to_yaml_file("mutatated_metamodel.yaml", exclude_defaults=False)

In [18]:
await core.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************
PLAY [node1] *******************************************************************


PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************
PLAY [node1] *******************************************************************


TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************
ok: [node1]
ok: [node1]
TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************

TASK [helm/prepare : Select official O-RAN SC M-release chart] ***********

In [19]:
metamodel, deployment_model = core.output()

deployment_model.to_yaml_file("deployment_model_4.yaml")
metamodel.to_yaml_file("metamodel_4.yaml")

# checkpoint 3: core deployed

In [20]:
from orchestrator.deployment import RICManagement
from orchestrator.models import DeploymentModel, MetaModel
from orchestrator.models.meta.ric import RICComponent, RICComponentType, RICImplementation

deployment_model = DeploymentModel.from_yaml_file("deployment_model_4.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_4.yaml")
metamodel.rics["osc"].components["chart_repository"] = RICComponent(
    type=RICComponentType.CHARTMUSEUM,
    implementation=RICImplementation.ORAN_SC,
)
metamodel.rics["osc"].components["xapp_onboarder"] = RICComponent(
    type=RICComponentType.XAPP_ONBOARDER,
    implementation=RICImplementation.ORAN_SC,
)

ric = RICManagement(metamodel=metamodel, deployment_model=deployment_model)

In [21]:
osc = ric.deployment_model.rics["osc"]
assert len(osc.components) == 9
assert {component.namespace for component in osc.components.values()} == {"ricplt"}
assert osc.components["database"].service_name == "service-ricplt-dbaas-tcp"
assert osc.components["chart_repository"].service_name == "oran-sc-osc-chart-repository"
assert osc.components["xapp_onboarder"].service_name == "oran-sc-osc-xapp-onboarder"
assert osc.components["a1_mediator"].service_name == "service-ricplt-a1mediator-http"
assert osc.components["e2_termination"].service_name == "service-ricplt-e2term-sctp-alpha"
assert osc.components["e2_termination"].ric_type_services["e2mgr"] == "service-ricplt-e2mgr-http"
# assert ric.mutatable_metamodel.rics["osc"].components["e2_termination"].config.e2.port == 36422

In [22]:
await ric.remove(force=True)


PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'oran-sc-osc-application-manager' Helm chart] ***

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'oran-sc-osc-chart-repository' Helm chart] ****

TASK [helm/uninstall : Uninstall 'oran-sc-osc-xapp-onboarder' Helm chart] ******

PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'oran-sc-osc-database' Helm chart] ************

PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'oran-sc-osc-e2-termination' Helm chart] ******

PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'oran-sc-osc-e2-manager' Helm chart] **********

PLAY [node1] 

In [23]:
# Prepare and install the complete RIC in dependency order.
# In particular, DBaaS and ChartMuseum must be ready before AppMgr.
await ric.deploy(force=True)


PLAY [node1] *******************************************************************
PLAY [node1] *******************************************************************


PLAY [node1] *******************************************************************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************
PLAY [node1] *******************************************************************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************


PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************
TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************


TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************
ok: [node1]
ok: 

In [24]:
osc = ric.deployment_model.rics["osc"]
assert all(component.is_installed for component in osc.components.values())
assert osc.components["e2_termination"].network_info is not None

In [25]:
metamodel, deployment_model = ric.output()

deployment_model.to_yaml_file("deployment_model_ric.yaml")
metamodel.to_yaml_file("metamodel_ric.yaml")

# checkpoint 4: OSC RIC deployed

In [26]:
from orchestrator.deployment import RANManagement
from orchestrator.models import DeploymentModel, MetaModel

deployment_model = DeploymentModel.from_yaml_file("deployment_model_ric.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_ric.yaml")

ran = RANManagement(metamodel=metamodel, deployment_model=deployment_model)

In [27]:
reconciled_metamodel, reconciled_deployment_model = ran.output()

reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_3.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_3.yaml")

In [28]:
await ran.remove(force=True)


PLAY [node1] *******************************************************************

TASK [helm/uninstall : Uninstall 'oai-gnb' Helm chart] *************************
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node1]

TASK [helm/uninstall : Wait for 'oai-gnb' pod to be deleted] *******************
ok: [node1]

TASK [helm/uninstall : Clean up Helm build directory for 'oai-gnb' chart if desired] ***
skipping: [node1]

PLAY RECAP *********************************************************************
node1                      : ok=2    changed=0    unreachable=0    failed=0    skipped=1    rescued=0    ignored=0   


In [29]:
await ran.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************
ok: [node1]

TASK [helm/prepare : Select Helm chart source] *********************************
ok: [node1]

TASK [helm/prepare : Remove stale 'oai-gnb' Helm build directory] **************
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node1]

TASK [helm/prepare : Create Helm build directory for 'oai-gnb' chart] **********
changed: [node1]

TASK [helm/prepare : Copy local 'oai-gnb' Helm chart directory for 'oai-gnb'] ***
changed: [node1]

TASK [helm/prepare : Render 'oai-gnb' Helm Chart.yaml file from Jinja template] ***
changed: [node1]

TASK [helm/prepa

In [30]:
metamodel, deployment_model = ran.output()

deployment_model.to_yaml_file("deployment_model_5.yaml")
metamodel.to_yaml_file("metamodel_5.yaml")

# checkpoint 5: ran deployed

In [31]:
from orchestrator.deployment import UEManagement
from orchestrator.models import DeploymentModel, MetaModel

deployment_model = DeploymentModel.from_yaml_file("deployment_model_5.yaml")
metamodel = MetaModel.from_yaml_file("metamodel_5.yaml")

ue = UEManagement(metamodel=metamodel, deployment_model=deployment_model)

In [32]:
reconciled_metamodel, reconciled_deployment_model = ue.output()

reconciled_deployment_model.to_yaml_file("reconciled_deployment_model_4.yaml")
reconciled_metamodel.to_yaml_file("reconciled_metamodel_4.yaml")

In [33]:
await ue.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Select official O-RAN SC M-release chart] *****************
ok: [node1]

TASK [helm/prepare : Select Helm chart source] *********************************
ok: [node1]

TASK [helm/prepare : Remove stale 'oai-sim-ue' Helm build directory] ***********
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node1]

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue' chart] *******
changed: [node1]

TASK [helm/prepare : Copy local 'oai-containerized' Helm chart directory for 'oai-sim-ue'] ***
changed: [node1]

TASK [helm/prepare : Render 'oai-sim-ue' Helm Chart.yaml file from Jinja template] ***
changed: [node1]



In [34]:
metamodel, deployment_model = ue.output()

deployment_model.to_yaml_file("deployment_model_6.yaml")
metamodel.to_yaml_file("metamodel_6.yaml")

In [35]:
# await ue.remove()

In [36]:
# await ran.remove(force=True)

In [37]:
# await ric.remove(force=True)

In [38]:
# await core.remove()

In [39]:
from pathlib import Path
from getpass import getpass

await ric.install_xapp(
    source_dir=Path("osc-xapps/hw-go"),
    ric_name="osc",
    wait_for_e2=True,
)


PLAY [node1] *******************************************************************

TASK [ric/xapp/install : Create remote xApp descriptor directory] **************
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node1]

TASK [ric/xapp/install : Copy xApp configuration descriptor] *******************
changed: [node1]

TASK [ric/xapp/install : Copy xApp controls schema] ****************************
changed: [node1]

TASK [ric/xapp/install : Onboard, install, register, and verify xApp] **********
changed: [node1]

PLAY RECAP *********************************************************************
node1                      : ok=4    changed=4    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   


## Third-party xApp interoperability test

This test uses WINES Lab's independent `kpm-basic-xapp`. Run every cell in this section in order. The preparation cell automatically clones the pinned source, builds the functional production image, verifies its startup command, pushes it to `docker.io/r2labuser`, and prepares the descriptor. The installation cell automatically replaces an older deployment using the non-functional public development image. The final audit verifies generic OSC onboarding, AppMgr registration, RMR routing, and a real KPM indication.


In [61]:
from orchestrator.deployment import RICManagement
from orchestrator.models import DeploymentModel, MetaModel

# Reload the deployed RIC state; rerunning the full deployment is not required.
third_party_deployment_model = DeploymentModel.from_yaml_file("deployment_model_ric.yaml")
third_party_metamodel = MetaModel.from_yaml_file("metamodel_ric.yaml")
ric = RICManagement(
    metamodel=third_party_metamodel,
    deployment_model=third_party_deployment_model,
)


In [62]:
from pathlib import Path
import getpass
import json
import platform
import shutil
import subprocess
import time

THIRD_PARTY_REPO = "https://github.com/wineslab/xDevSM-xapps-examples.git"
THIRD_PARTY_REF = "4fc2f1fc60a29190521b7b7056d6828828ef7249"
SOURCE_DIR = Path("osc-xapps/xdevsm-examples").resolve()
DESCRIPTOR_DIR = Path("osc-xapps/kpm-basic-xapp").resolve()
BUILD_AND_PUSH_IMAGE = True  # Keep enabled: this section is intentionally self-contained.
PRIVATE_IMAGE = "docker.io/r2labuser/kpm-basic-xapp:0.3.0-orchestrator7"

if not (SOURCE_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", THIRD_PARTY_REPO, str(SOURCE_DIR)], check=True)
subprocess.run(["git", "-C", str(SOURCE_DIR), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(SOURCE_DIR), "checkout", THIRD_PARTY_REF], check=True)
subprocess.run(["git", "-C", str(SOURCE_DIR), "config", "submodule.xDevSM.url", "https://github.com/wineslab/xDevSM.git"], check=True)
subprocess.run(["git", "-C", str(SOURCE_DIR), "submodule", "update", "--init", "--recursive"], check=True)

# Normalize the third-party runtime configuration before building the image.
# The upstream example is pinned to its author's ricxapp-j namespace; both the
# mounted descriptor and the files embedded in the image must target ricxapp.
source_config_path = SOURCE_DIR / "kpm_basic_xapp/config/config-file.json"
source_config = json.loads(source_config_path.read_text())
source_config["APP_NAMESPACE"] = "ricxapp"
source_config["controls"]["subscription"]["url"] = "http://service-ricxapp-kpm-basic-xapp-http.ricxapp:8080/"
source_config["controls"]["subscription"]["clientEndpoint"] = "service-ricxapp-kpm-basic-xapp-http.ricxapp"
source_config_path.write_text(json.dumps(source_config, indent=2) + "\n")
seed_route_path = SOURCE_DIR / "kpm_basic_xapp/config/uta_rtg.rt"
seed_routes = seed_route_path.read_text().replace(
    "service-ricxapp-j-kpm-xapp-rmr.ricxapp-j",
    "service-ricxapp-kpm-basic-xapp-rmr.ricxapp",
)
seed_route_path.write_text(seed_routes)
if "ricxapp-j" in source_config_path.read_text() or "ricxapp-j" in seed_route_path.read_text():
    raise RuntimeError("Legacy ricxapp-j references remain in the KPM xApp runtime configuration")

# The upstream handler blocks inside RMRXapp.__init__ waiting for a seed route
# that points back to the same not-yet-started xApp. Allow initialization to
# complete; RtMgr pushes the authoritative routes before the subscription POST.
handler_path = SOURCE_DIR / "xDevSM/handlers/xDevSM_rmr_xapp.py"
handler_text = handler_path.read_text().replace(
    "rmr_wait_for_ready=True",
    "rmr_wait_for_ready=False",
)
handler_text = handler_text.replace(
    "post_init=self._post_init, rmr_wait_for_ready=False)",
    "post_init=self._post_init, rmr_wait_for_ready=False, use_fake_sdl=True)",
)
handler_path.write_text(handler_text)
if "rmr_wait_for_ready=False, use_fake_sdl=True" not in handler_path.read_text():
    raise RuntimeError("Failed to disable the xApp's obsolete RMR/SDL startup waits")

# Replace legacy SDL/protobuf node discovery with the M-release E2Mgr REST API.
method_start = handler_text.index("    def get_selected_e2node_info(")
method_end = handler_text.index("    def loading_ports(", method_start)
rest_discovery_lines = [
    "    def get_selected_e2node_info(self, e2node_inventory_name=None):",
    "        from types import SimpleNamespace",
    "",
    "        states_uri = self.e2mgr_link + 'states'",
    "        self.logger.info('[xDevSMRMRXapp] discovering E2 nodes through {}'.format(states_uri))",
    "        response = requests.get(states_uri, timeout=10)",
    "        response.raise_for_status()",
    "        connected_nodes = [node for node in response.json()",
    "                           if node.get('connectionStatus') == 'CONNECTED'",
    "                           and (not e2node_inventory_name or node.get('inventoryName') == e2node_inventory_name)]",
    "        if not connected_nodes:",
    "            self.logger.error('[xDevSMRMRXapp] no matching connected E2 node available')",
    "            return None, None",
    "",
    "        inventory_name = connected_nodes[0]['inventoryName']",
    "        selected_gnb = SimpleNamespace(inventory_name=inventory_name)",
    "        gnb_info = self.get_ran_info(selected_gnb)",
    "        self.logger.info('[xDevSMRMRXapp] selected gnb {}'.format(inventory_name))",
    "        return selected_gnb, gnb_info",
    "",
]
rest_discovery_method = "\n".join(rest_discovery_lines) + "\n"
handler_text = handler_text[:method_start] + rest_discovery_method + handler_text[method_end:]
handler_path.write_text(handler_text)
if "discovering E2 nodes through" not in handler_path.read_text():
    raise RuntimeError("Failed to install M-release REST-based E2 node discovery")

# Emit unbuffered lifecycle markers so functional failures are immediately
# attributable even when the third-party MDC logger suppresses Python records.
main_path = SOURCE_DIR / "kpm_basic_xapp/kpm_xapp.py"
main_text = main_path.read_text()
main_text = main_text if "[KPM-BOOT] constructing xApp framework" in main_text else main_text.replace(
    "    # Creating a generic xDevSM RMR xApp\n",
    "    print('[KPM-BOOT] constructing xApp framework', flush=True)\n    # Creating a generic xDevSM RMR xApp\n",
)
main_text = main_text if "[KPM-BOOT] framework constructed" in main_text else main_text.replace(
    "    # Adding kpm functionalities to the xapp\n",
    "    print('[KPM-BOOT] framework constructed', flush=True)\n    # Adding kpm functionalities to the xapp\n",
)
main_text = main_text if "[KPM-BOOT] discovering connected gNB" in main_text else main_text.replace(
    "    gnb, gnb_info = xapp_gen.get_selected_e2node_info(args.gnb_target)\n",
    "    print('[KPM-BOOT] discovering connected gNB', flush=True)\n    gnb, gnb_info = xapp_gen.get_selected_e2node_info(args.gnb_target)\n    print('[KPM-BOOT] gNB discovery completed', flush=True)\n",
)
main_text = main_text if "[KPM-BOOT] submitting KPM subscription" in main_text else main_text.replace(
    "    kpm_xapp.subscribe(gnb=gnb, ev_trigger=ev_trigger_tuple, func_def=func_def_sub_dict,  ran_period_ms=1000, sst=args.sst, sd=args.sd)\n",
    "    print('[KPM-BOOT] submitting KPM subscription', flush=True)\n    kpm_xapp.subscribe(gnb=gnb, ev_trigger=ev_trigger_tuple, func_def=func_def_sub_dict,  ran_period_ms=1000, sst=args.sst, sd=args.sd)\n    print('[KPM-BOOT] subscription POST completed', flush=True)\n",
)
# Prefer KPM action-definition format 1 for this platform smoke test.
# Upstream prefers format 4, which filters by NSSAI and emits nothing when
# no UE matches the requested slice. Format 1 reports node-level metrics.
format4_selector = "    if len(func_def_dict[format_action_def_e.FORMAT_4_ACTION_DEFINITION]) == 0:\n        selected_format = format_action_def_e.FORMAT_1_ACTION_DEFINITION\n    else:\n        selected_format = format_action_def_e.FORMAT_4_ACTION_DEFINITION"
format1_selector = "    if len(func_def_dict[format_action_def_e.FORMAT_1_ACTION_DEFINITION]) > 0:\n        selected_format = format_action_def_e.FORMAT_1_ACTION_DEFINITION\n    elif len(func_def_dict[format_action_def_e.FORMAT_4_ACTION_DEFINITION]) > 0:\n        selected_format = format_action_def_e.FORMAT_4_ACTION_DEFINITION"
main_text = main_text.replace(format4_selector, format1_selector)
main_path.write_text(main_text)
if format1_selector not in main_path.read_text():
    raise RuntimeError("Failed to configure UE-independent KPM format 1")
if "[KPM-BOOT] subscription POST completed" not in main_path.read_text():
    raise RuntimeError("Failed to add deterministic KPM lifecycle diagnostics")

# xDevSM's standard platform namespace is ricplt, which now matches this deployment.
# AppMgr registration is owned by the orchestrator. Disable the framework's duplicate
# registration thread, which depends on legacy Kubernetes service-link variables.
dockerfile = SOURCE_DIR / "docker/Dockerfile.kpm_basic_xapp"
dockerfile_text = dockerfile.read_text()
registration_patch = "RUN sed -i 's/self._appthread = Thread(target=self.registerXapp).start()/self._appthread = None/' /usr/local/lib/python3.8/site-packages/ricxappframe/xapp_frame.py && grep -q 'self._appthread = None' /usr/local/lib/python3.8/site-packages/ricxappframe/xapp_frame.py"
if registration_patch not in dockerfile_text:
    dockerfile.write_text(dockerfile_text + "\n# Registration is performed by the orchestrator.\n" + registration_patch + "\n")

image = PRIVATE_IMAGE
if BUILD_AND_PUSH_IMAGE:
    if subprocess.run(["docker", "info"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode != 0:
        if platform.system() != "Darwin":
            raise RuntimeError("Docker is installed but its daemon is not running.")
        print("Starting Docker Desktop...")
        subprocess.run(["open", "-a", "Docker"], check=True)
        for _ in range(90):
            if subprocess.run(["docker", "info"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0:
                break
            time.sleep(2)
        else:
            raise RuntimeError("Docker Desktop did not become ready within three minutes.")
    credential_helper = shutil.which("docker-credential-desktop")
    credential_ok = False
    if credential_helper:
        credential_ok = subprocess.run([credential_helper, "get"], input="https://index.docker.io/v1/", text=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
    if not credential_ok:
        print("Docker Hub login is required once for r2labuser. Use a Docker Hub access token, not the account password.")
        docker_token = getpass.getpass("Docker Hub access token: ")
        subprocess.run(["docker", "login", "--username", "r2labuser", "--password-stdin"], input=docker_token + "\n", text=True, check=True)
    inspect_result = subprocess.run(["docker", "inspect", image, "--format", "{{json .Config.Cmd}}"], capture_output=True, text=True)
    image_cmd = inspect_result.stdout.strip() if inspect_result.returncode == 0 else ""
    if "kpm_xapp.py" not in image_cmd:
        subprocess.run(["docker", "build", "--platform", "linux/amd64", "--pull", "-f", "docker/Dockerfile.kpm_basic_xapp", "-t", image, "."], cwd=SOURCE_DIR, check=True)
        image_cmd = subprocess.run(["docker", "inspect", image, "--format", "{{json .Config.Cmd}}"], check=True, capture_output=True, text=True).stdout.strip()
    else:
        print(f"Reusing the existing functional image {image}")
    if "kpm_xapp.py" not in image_cmd:
        raise RuntimeError(f"Built image has an unexpected startup command: {image_cmd}")
    print(f"Verified image startup command: {image_cmd}")
    for push_attempt in range(1, 4):
        push_result = subprocess.run(["docker", "push", image])
        if push_result.returncode == 0:
            break
        if push_attempt == 3:
            raise RuntimeError(f"Docker Hub push failed after {push_attempt} attempts.")
        print(f"Push attempt {push_attempt} failed; retrying in 10 seconds...")
        time.sleep(10)

DESCRIPTOR_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(SOURCE_DIR / "kpm_basic_xapp/config/schema.json", DESCRIPTOR_DIR / "schema.json")
descriptor = json.loads((SOURCE_DIR / "kpm_basic_xapp/config/config-file.json").read_text())
descriptor["version"] = "0.3.7"
descriptor["APP_NAMESPACE"] = "ricxapp"
descriptor["controls"]["subscription"]["url"] = "http://service-ricxapp-kpm-basic-xapp-http.ricxapp:8080/"
descriptor["controls"]["subscription"]["clientEndpoint"] = "service-ricxapp-kpm-basic-xapp-http.ricxapp"
registry_and_name, tag = image.rsplit(":", 1)
registry, name = registry_and_name.split("/", 1)
descriptor["containers"][0]["image"] = {"registry": registry, "name": name, "tag": tag}
(DESCRIPTOR_DIR / "config-file.json").write_text(json.dumps(descriptor, indent=2) + "\n")
print(f"Prepared {descriptor['name']} {descriptor['version']} using {image}")


HEAD is now at 4fc2f1f fixing xdevsm


M	docker/Dockerfile.kpm_basic_xapp
M	kpm_basic_xapp/config/config-file.json
M	kpm_basic_xapp/config/uta_rtg.rt
M	kpm_basic_xapp/kpm_xapp.py
M	xDevSM


#0 building with "desktop-linux" instance using docker driver

#1 [internal] load build definition from Dockerfile.kpm_basic_xapp
#1 transferring dockerfile: 3.17kB done
#1 WARN: FromAsCasing: 'as' and 'FROM' keywords' casing do not match (line 1)
#1 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.8-slim
#2 ...

#3 [auth] library/python:pull token for registry-1.docker.io
#3 DONE 0.0s

#2 [internal] load metadata for docker.io/library/python:3.8-slim
#2 DONE 0.9s

#4 [internal] load .dockerignore
#4 transferring context: 2B done
#4 DONE 0.0s

#5 [stretch  1/10] FROM docker.io/library/python:3.8-slim@sha256:1d52838af602b4b5a831beb13a0e4d073280665ea7be7f69ce2382f29c5a613f
#5 resolve docker.io/library/python:3.8-slim@sha256:1d52838af602b4b5a831beb13a0e4d073280665ea7be7f69ce2382f29c5a613f done
#5 DONE 0.0s

#6 [internal] load build context
#6 transferring context: 30.50kB done
#6 DONE 0.0s

#7 [stretch  4/10] RUN wget -nv --content-disposition https://packagecloud.io/

Verified image startup command: ["python","kpm_xapp.py"]
The push refers to repository [docker.io/r2labuser/kpm-basic-xapp]
ba2e91e883af: Waiting
5c9152b55064: Waiting
c427acd31828: Waiting
18f3e1276302: Waiting
aa69a2493b80: Waiting
350302e74389: Waiting
1f39e038687f: Waiting
4f4fb700ef54: Waiting
cfa30fec0699: Waiting
68047ba831c3: Waiting
030d7bdc20a6: Waiting
508fd9d66243: Waiting
2a3d8c7c952c: Waiting
d1d4dbc3a4e8: Waiting
e03486f6265f: Waiting
08706d2f8e7c: Waiting
302e3ee49805: Waiting
a3f1dfe736c5: Waiting
3971691a3637: Waiting
508fd9d66243: Waiting
2a3d8c7c952c: Waiting
d1d4dbc3a4e8: Waiting
e03486f6265f: Waiting
68047ba831c3: Waiting
030d7bdc20a6: Waiting
08706d2f8e7c: Waiting
302e3ee49805: Waiting
a3f1dfe736c5: Waiting
3971691a3637: Waiting
5c9152b55064: Waiting
c427acd31828: Waiting
18f3e1276302: Waiting
aa69a2493b80: Waiting
ba2e91e883af: Waiting
350302e74389: Waiting
1f39e038687f: Waiting
4f4fb700ef54: Waiting
cfa30fec0699: Waiting
508fd9d66243: Waiting
2a3d8c7c952c: Wait

In [63]:
await ric.install_xapp(
    source_dir=DESCRIPTOR_DIR,
    ric_name="osc",
    wait_for_e2=True,
)



PLAY [node1] *******************************************************************

TASK [ric/xapp/install : Create remote xApp descriptor directory] **************
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node1]

TASK [ric/xapp/install : Copy xApp configuration descriptor] *******************
changed: [node1]

TASK [ric/xapp/install : Copy xApp controls schema] ****************************
ok: [node1]

TASK [ric/xapp/install : Onboard, install, register, and verify xApp] **********
changed: [node1]

PLAY RECAP *********************************************************************
node1                      : ok=4    changed=2    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   


In [65]:
# Deterministic platform audit: deployment, AppMgr registration, RMR routing,
# and E2 connectivity. Live indications additionally require compatible KPM
# capabilities and active RAN/UE measurements, so that check is opt-in.
RUN_LIVE_KPM_INDICATION_TEST = True
await ric.audit_xapp(
    xapp_name="kpm-basic-xapp",
    ric_name="osc",
    indication_timeout=180 if RUN_LIVE_KPM_INDICATION_TEST else 0,
)



PLAY [node1] *******************************************************************

TASK [ric/xapp/audit : Audit xApp platform and functional integration] *********
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [node1]

PLAY RECAP *********************************************************************
node1                      : ok=1    changed=0    unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   


In [ ]:
# await prov.unprovision()